# 🧠 Invoice Assistant: Complete Backend A-Z

This notebook contains the **entire backend logic** for the Invoice Assistant from scratch. It is organized into logical layers, from infrastructure to delivery.

### 🟢 Layer 1: Infrastructure & Environment Setup
Handles configuration and LLM initialization.

In [ ]:
import os
import logging
import re
import json
import asyncio
from datetime import datetime
from typing import List, Optional, Set
from tempfile import NamedTemporaryFile
from dotenv import load_dotenv

def init_config():
    load_dotenv()
    logging.basicConfig(level=logging.INFO)

def get_env(key: str, default: str = None) -> str:
    return os.getenv(key, default)

# LLM Engine Factory
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def get_llm():
    if get_env("AZURE_OPENAI_API_KEY"):
        return AzureChatOpenAI(
            azure_endpoint=get_env("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=get_env("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version=get_env("AZURE_OPENAI_API_VERSION"),
            temperature=0, request_timeout=60
        )
    if get_env("OPENAI_API_KEY"):
        return ChatOpenAI(model="gpt-4o", temperature=0, request_timeout=60)
    if get_env("GOOGLE_API_KEY"):
        return ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0, request_timeout=60)
    raise ValueError("Missing API Keys")

### 🟡 Layer 2: Data Models
Defines the structured schema for extracted data.

In [ ]:
from pydantic import BaseModel, Field

class InvoiceData(BaseModel):
    supplier: Optional[str] = Field(None)
    invoice_date: Optional[str] = Field(None)
    reference: Optional[str] = Field(None)
    posting_date: Optional[str] = Field(None)
    amount: Optional[str] = Field(None)
    tax_amount: Optional[str] = Field(None)
    validation_error: Optional[str] = Field(None)

class InvoiceList(BaseModel):
    invoices: List[InvoiceData]

### 🟠 Layer 3: Extraction & AI Parsing
Handles PDF processing, OCR fallback, and structured AI extraction.

In [ ]:
from pypdf import PdfReader
from pdf2image import convert_from_path
import pytesseract
import shutil
from langchain_core.prompts import ChatPromptTemplate

def extract_text(pdf_path: str) -> str:
    text = ""
    try:
        reader = PdfReader(pdf_path)
        text = "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
    except: pass

    if len(text.strip()) < 50:
        if shutil.which("pdftoppm") and shutil.which("tesseract"):
            images = convert_from_path(pdf_path)
            text = "\n".join([pytesseract.image_to_string(img) for img in images])
    return text

def analyze_text(text: str) -> List[dict]:
    llm = get_llm().with_structured_output(InvoiceList)
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a production-grade SAP extraction agent. Extract: supplier, invoice_date, reference, posting_date, amount, tax_amount.\n"
            "- posting_date: Leave empty.\n"
            "Format dates as DD.MM.YYYY. Ensure amounts use '.' as decimal separator."
        )),
        ("human", "{text}")
    ])
    result = (prompt | llm).invoke({"text": text})
    return [inv.model_dump() for inv in result.invoices]

### 🔵 Layer 4: Validation & Logic Control
Standardizes numbers and enforces the Posting Date = Today rule.

In [ ]:
def clean_numeric(val: str) -> str:
    if not val: return "0.00"
    s = str(val).strip()
    if ',' in s and '.' in s:
        if s.rfind(',') > s.rfind('.'): s = s.replace('.', '').replace(',', '.')
        else: s = s.replace(',', '')
    elif ',' in s: s = s.replace(',', '.')
    s = re.sub(r"[^\d.]", "", s)
    return s if s else "0.00"

def validate_invoice(data: dict) -> dict:
    data["amount"] = clean_numeric(data.get("amount"))
    data["tax_amount"] = clean_numeric(data.get("tax_amount"))
    
    # Force Posting Date to TODAY
    data["posting_date"] = datetime.now().strftime("%d.%m.%Y")
    return data

def detect_unique_invoices(invoices: List[dict]) -> List[dict]:
    final, seen = [], set()
    for inv in invoices:
        key = f"{inv.get('supplier')}_{inv.get('reference')}".lower()
        if key not in seen:
            seen.add(key)
            final.append(inv)
    return final

### 🔘 Layer 5: State Management (Session Store)
Persistently stores pending invoices to disk.

In [ ]:
STORAGE_FILE = "session_data.json"

class SessionStore:
    def __init__(self):
        self._pending_invoices = []
        self._load()

    def _load(self):
        if os.path.exists(STORAGE_FILE):
            with open(STORAGE_FILE, "r") as f: self._pending_invoices = json.load(f).get("invoices", [])

    def _save(self):
        with open(STORAGE_FILE, "w") as f: json.dump({"invoices": self._pending_invoices}, f, indent=2)

    def add_invoices(self, invoices: List[dict]):
        existing = {f"{i.get('supplier')}_{i.get('reference')}".lower() for i in self._pending_invoices}
        for inv in invoices:
            if f"{inv.get('supplier')}_{inv.get('reference')}".lower() not in existing: 
                self._pending_invoices.append(inv)
        self._save()

session_store = SessionStore()

### 🏁 Layer 6: Orchestration & Delivery (API)
The final entry point that ties everything together.

In [ ]:
def process_invoice(path: str):
    text = extract_text(path)
    invoices = analyze_text(text)
    unique = detect_unique_invoices(invoices)
    return [validate_invoice(inv) for inv in unique]

# FastAPI Delivery Layer
from fastapi import FastAPI, UploadFile, File

app = FastAPI()

@app.post("/upload")
async def upload_invoices(files: List[UploadFile] = File(...)):
    results = []
    for file in files:
        with NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(await file.read())
            results.extend(process_invoice(tmp.name))
    session_store.add_invoices(results)
    return results